# Atlanta Shopping Malls: Advanced Economic Models

**Research sample:** 24 ZCTAs (30322 excluded), 3 destinations, ACS 2019–2024 5-year estimates.

**Important:** All findings are associations, not causal impacts. The 2019–2024 ACS vintages overlap, so panel results are exploratory. Historical income is not analyzed as a nominal change because ACS income dollars differ across vintages.

Run from the extracted folder. Files are read from `inputs/`; results saved in `outputs/`.

**Models:** linear and quadratic distances; piecewise 3/5/8 mile thresholds (only if sample supports); mall-specific and mall-type interactions with ZIP-clustered standard errors; distance-decay accessibility; historical Gini near×post with ZIP/year fixed effects; change-vs-distance; controls-ready extension.

**Caution:** With only 24 unique ZCTAs, model selection and multiple comparisons require restraint. Phipps and Lenox share nearby neighborhoods. Do not interpret pooled rows as independent.

In [1]:
%pip install pandas numpy scipy statsmodels matplotlib


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
BASE=Path.cwd()
if not (BASE/'inputs'/'acs_2019_2024_24_zctas.csv').exists():
    raise FileNotFoundError('Open the extracted Atlanta_Advanced_Models folder in VS Code, then run this notebook from that folder. Current working directory: '+str(BASE))
OUT=BASE/'outputs';OUT.mkdir(exist_ok=True)

In [3]:
acs=pd.read_csv(BASE/'inputs'/'acs_2019_2024_24_zctas.csv',dtype={'ZCTA':str});routes=pd.read_csv(BASE/'inputs'/'routes_2024_merged.csv',dtype={'ZCTA':str})[['ZCTA','Destination','driving_miles']]
base=routes.merge(acs[acs.year==2024][['ZCTA','income','gini']],on='ZCTA',validate='many_to_one')
rows=[]
def fit(df,formula,name,mall='all',cluster=False):
 d=df.dropna(subset=[c for c in [formula.split('~')[0].strip(), 'driving_miles' if 'driving_miles' in formula else ''] if c in df.columns]).copy()
 try:
  m=smf.ols(formula,data=d).fit(cov_type='cluster',cov_kwds={'groups':d.ZCTA}) if cluster else smf.ols(formula,data=d).fit(cov_type='HC3')
  for k in m.params.index:
   if k!='Intercept':rows.append(dict(model=name,mall=mall,term=k,n=len(d),r2=m.rsquared,adj_r2=m.rsquared_adj,coef=m.params[k],se=m.bse[k],p=m.pvalues[k],aic=m.aic))
  return m
 except Exception as e:print('FAILED',name,mall,str(e));return None
for mall,d in base.groupby('Destination'):
 for outcome in ['income','gini']:
  dd=d.dropna(subset=[outcome]).copy()
  if outcome=='income':dd['log_income']=np.log(dd.income);y='log_income'
  else:y='gini'
  dd['d2']=dd.driving_miles**2
  fit(dd,f'{y} ~ driving_miles','linear',mall)
  fit(dd,f'{y} ~ driving_miles + d2','quadratic',mall)
  for knot in [3,5,8]:
   if (dd.driving_miles<=knot).sum()>=5 and (dd.driving_miles>knot).sum()>=5:
    dd['hinge']=np.maximum(0,dd.driving_miles-knot)
    fit(dd,f'{y} ~ driving_miles + hinge',f'piecewise_{knot}mi',mall)
# pooled interactions, use cluster robust SE due to repeated ZCTAs
base['log_income']=np.log(base.income)
for y in ['log_income','gini']:
 fit(base,f'{y} ~ driving_miles * C(Destination)',f'pooled_mall_interaction_{y}',cluster=True)
# Buckhead vs PCM, use mean Buckhead distance per ZCTA to avoid pseudo-replication
wide=routes.pivot(index='ZCTA',columns='Destination',values='driving_miles');wide['Buckhead_mean']=(wide['Lenox Square']+wide['Phipps Plaza'])/2
long=pd.concat([wide[['Buckhead_mean']].rename(columns={'Buckhead_mean':'driving_miles'}).assign(mall_type='Buckhead cluster'),wide[['Ponce City Market']].rename(columns={'Ponce City Market':'driving_miles'}).assign(mall_type='Mixed use')]).reset_index().merge(acs[acs.year==2024][['ZCTA','income','gini']],on='ZCTA');long['log_income']=np.log(long.income)
for y in ['log_income','gini']:fit(long,f'{y} ~ driving_miles * C(mall_type)',f'mall_type_interaction_{y}',cluster=True)
# accessibility index, transparent distance-decay scale (5 mi)
access=wide[['Lenox Square','Phipps Plaza','Ponce City Market']].copy()
access['accessibility_5mi']=np.exp(-access/5).sum(axis=1)
access['nearest_miles']=access[['Lenox Square','Phipps Plaza','Ponce City Market']].min(axis=1)
access=access.reset_index().merge(acs[acs.year==2024][['ZCTA','income','gini']],on='ZCTA');access['log_income']=np.log(access.income)
for y in ['log_income','gini']:
 fit(access,f'{y} ~ accessibility_5mi',f'accessibility_{y}')
 fit(access,f'{y} ~ nearest_miles',f'nearest_mall_{y}')
# historical descriptive panel: use Gini, ZIP and vintage FE; 2022 post flag, cluster ZIP
hist=acs.merge(wide[['Phipps Plaza','Ponce City Market']].reset_index(),on='ZCTA')
for mall in ['Phipps Plaza','Ponce City Market']:
 dd=hist.copy();dd['near']=(dd[mall]<=dd[mall].median()).astype(int);dd['post']=(dd.year>=2022).astype(int);dd['near_post']=dd.near*dd.post
 fit(dd,'gini ~ near_post + C(ZCTA) + C(year)',f'exploratory_panel_near_post_{mall}',cluster=True)
# distance x change 2019-2024, gini only
piv=acs.pivot(index='ZCTA',columns='year',values='gini');change=(piv[2024]-piv[2019]).rename('gini_change').reset_index()
for mall in ['Lenox Square','Phipps Plaza','Ponce City Market']:
 d=change.merge(routes[routes.Destination==mall],on='ZCTA');fit(d,'gini_change ~ driving_miles','gini_change_2019_2024',mall)
res=pd.DataFrame(rows);res['fdr_bh_p']=multipletests(res.p.fillna(1),method='fdr_bh')[1];res.to_csv(OUT/'all_model_coefficients.csv',index=False)
base.to_csv(OUT/'cross_section_2024.csv',index=False);access.to_csv(OUT/'accessibility_by_zcta.csv',index=False);hist.to_csv(OUT/'historical_panel.csv',index=False)


In [4]:
print('Models run:',res.model.nunique())
display(res.sort_values('p').head(25))
display(res.query("term == 'near_post' or term == 'accessibility_5mi'"))

Models run: 16


,model,mall,term,n,r2,adj_r2,coef,se,p,aic,fdr_bh_p
75,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30318],144,0.848840,0.810388,0.037017,1.034379e-15,0.000000e+00,-767.125937,0.000000e+00
61,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30303],144,0.848840,0.810388,0.062567,1.158771e-15,0.000000e+00,-767.125937,0.000000e+00
71,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30314],144,0.848840,0.810388,0.021350,1.025406e-15,0.000000e+00,-767.125937,0.000000e+00
70,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30313],144,0.848840,0.810388,-0.019217,1.143976e-15,0.000000e+00,-767.125937,0.000000e+00
69,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30312],144,0.848840,0.810388,0.040167,1.108172e-15,0.000000e+00,-767.125937,0.000000e+00
96,exploratory_panel_near_post_Ponce City Market,all,C(ZCTA)[T.30310],144,0.843858,0.804138,0.021717,1.832306e-15,0.000000e+00,-762.456123,0.000000e+00
97,exploratory_panel_near_post_Ponce City Market,all,C(ZCTA)[T.30311],144,0.843858,0.804138,0.045217,1.581531e-15,0.000000e+00,-762.456123,0.000000e+00
74,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30317],144,0.848840,0.810388,-0.018167,1.137839e-15,0.000000e+00,-767.125937,0.000000e+00
68,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30311],144,0.848840,0.810388,0.045217,1.317236e-15,0.000000e+00,-767.125937,0.000000e+00
67,exploratory_panel_near_post_Phipps Plaza,all,C(ZCTA)[T.30310],144,0.848840,0.810388,0.021717,1.032565e-15,0.000000e+00,-767.125937,0.000000e+00


,model,mall,term,n,r2,adj_r2,coef,se,p,aic,fdr_bh_p
56,accessibility_log_income,all,accessibility_5mi,23,0.222206,0.185168,0.394144,0.164217,0.016389,16.909493,0.040471
58,accessibility_gini,all,accessibility_5mi,24,0.021128,-0.023367,-0.010450,0.015657,0.504468,-95.789946,0.789892
88,exploratory_panel_near_post_Phipps Plaza,all,near_post,144,0.848840,0.810388,-0.011392,0.008254,0.167530,-767.125937,0.343578
117,exploratory_panel_near_post_Ponce City Market,all,near_post,144,0.843858,0.804138,0.005558,0.008574,0.516814,-762.456123,0.791576


## Optional ACS controls extension
Place `acs_controls_2024.csv` inside `inputs/` with columns `ZCTA, poverty_rate, bachelor_share, median_home_value` (at least one control required). **Do not insert fabricated values.** The next cell will skip until you supply it. Controls can be highly correlated with income; interpret adjusted associations cautiously. Poverty rate and education can themselves be outcomes or mediators, so select controls based on the research question.

In [5]:
controls_file=BASE/'inputs'/'acs_controls_2024.csv'
if controls_file.exists():
    controls=pd.read_csv(controls_file,dtype={'ZCTA':str})
    controls['ZCTA']=controls.ZCTA.str.zfill(5)
    cols=[c for c in ['poverty_rate','bachelor_share','median_home_value'] if c in controls.columns]
    if not cols: raise ValueError('No expected controls columns found')
    adjusted=base.merge(controls[['ZCTA']+cols],on='ZCTA',validate='many_to_one')
    adjusted['log_income']=np.log(adjusted.income)
    for y in ['log_income','gini']:
        for mall,d in adjusted.groupby('Destination'):
            use=[c for c in cols if d[c].notna().sum()>=18]
            if use:
                f=y+' ~ driving_miles + '+' + '.join('np.log(median_home_value)' if c=='median_home_value' else c for c in use)
                print('Adjusted:',mall,y,f)
                display(smf.ols(f,data=d).fit(cov_type='HC3').summary())
else:
    print('Optional controls not yet uploaded. All existing models are complete.')

Optional controls not yet uploaded. All existing models are complete.


## Interpretation guardrails
- Piecewise knots are **pre-specified candidates**, not discovered causal boundaries; verify observations on each side.
- Compare adjusted R²/AIC cautiously; coefficients from different outcomes are not directly comparable.
- `fdr_bh_p` corrects the full collection of coefficient tests and is conservative/exploratory, not a substitute for pre-registration.
- Pooled interaction model uses ZIP-clustered errors; repeated destination observations are not independent.
- Gini fixed-effects near×post uses **ACS release vintages** with overlapping underlying years; not a valid clean event-study/DID causal design.
- Historical income comparisons need CPI normalization before interpretation.
- Geographic spatial autocorrelation, residential sorting, travel-time variation and endogeneity remain unaddressed.
- The 30322 ZCTA is excluded; 30303 income is missing in 2024 but its Gini is included.